# Avenue 2 (reference only): the same multicollinearity screen, applied read-only

**This notebook makes NO changes to Avenue 2.** It does not edit
`models/growth_curve_attribution/broad_environmental_check.py`, does not retrain any Avenue 2
model, and does not touch any Avenue 2 output file. It only imports Avenue 2's existing
`FEATURE_GROUPS`/`SCOPE_GROUPS` definitions to compare them against what the shared screen
(`models/xgb_environmental/multicollinearity_screen.py`, the same module the companion
`multicollinearity_screen_av1.ipynb` notebook uses) would independently suggest.

**Why a separate notebook rather than one shared notebook.** Avenue 1 and Avenue 2 answer
different questions (per-plot height prediction vs. per-plot growth-curve-deviation
attribution) with different architectures, so their named feature-set tiers are genuinely
different objects, not the same tiers under two names -- e.g. Avenue 2 treats `management`
(`CanopyCover`, `Thin`, etc.) as a candidate comparison group (`SCOPE_GROUPS["...plus_management"]`),
because its target model doesn't already consume those columns elsewhere the way Avenue 1's
DNN/PINN main network does. Building one merged notebook would blur that real architectural
difference rather than explain it.

**Scope.** Same as the Avenue 1 notebook: only the environmental candidate pool (terrain, wind,
climate, soil, forest-edge) -- not `management`, which is already in the dataset as a raw survey
field and is Avenue 2's own separate, deliberate architectural comparison axis, not a
multicollinearity question this screen is built to answer.

In [1]:
import sys
from pathlib import Path

notebook_directory = Path.cwd().resolve()
project_root = next(
    folder for folder in [notebook_directory, *notebook_directory.parents]
    if (folder / "README.md").exists() and (folder / "data").exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)

# Read-only imports -- Avenue 2's own, real, already-existing feature-group definitions.
from models.growth_curve_attribution.broad_environmental_check import (
    FEATURE_GROUPS,
    SCOPE_GROUPS,
    prepare_broad_table,
    columns_for_groups,
)
from models.growth_curve_attribution.scale_comparison_check import TARGET
from models.growth_curve_attribution.explain_signal import FINAL_FEATURE_COLUMNS
from models.growth_curve_attribution.run_wind_height_swap_check import SWAPPED_COLUMNS
from models.xgb_environmental.xgb_environmental import FEATURE_PROVENANCE, TERRAIN_AND_WIND_COLUMNS
from models.xgb_environmental.multicollinearity_screen import run_full_screen

print("Avenue 2's own FEATURE_GROUPS:")
for name, columns in FEATURE_GROUPS.items():
    print(f"  {name:<14} {len(columns):>3} columns")
print(f"\nAvenue 2's own terrain_wind (FINAL_FEATURE_COLUMNS): {len(FINAL_FEATURE_COLUMNS)} columns")

Avenue 2's own FEATURE_GROUPS:
  terrain_wind    17 columns
  climate          5 columns
  soil_site        5 columns
  edge_position    6 columns
  management       5 columns

Avenue 2's own terrain_wind (FINAL_FEATURE_COLUMNS): 17 columns


## 1. Target-circularity check (same standard as Avenue 1)

Avenue 2's target, `local_y_max_difference` (`y_max_fit - y_max_yldc`), is built in
`models/growth_curve_attribution/scale_comparison_check.py::build_plot_level_table()` purely
from height/Age survey observations and the Forestry Commission's static yield-class lookup --
confirmed by reading that function directly. No environmental variable (terrain, wind, climate,
soil, or edge) appears anywhere in its construction. **Pass -- no circularity**, same conclusion
as the Avenue 1 notebook, independently re-confirmed here for this avenue's own target.

In [2]:
cohort = "4survey"
non_management_groups = ["terrain_wind", "climate", "soil_site", "edge_position"]
raw_candidate_columns = columns_for_groups(non_management_groups)

# Same exclusion the Avenue 1 notebook applies: unordered categorical class IDs aren't valid
# input to a Spearman correlation/VIF screen (they get one-hot-encoded by prepare_broad_table
# below, which is the right treatment for FITTING a model, but not for THIS screen).
CATEGORICAL_COLUMNS = ["ceh_pedotope", "ceh_subsurface_drainage", "ceh_textural_composition"]
candidate_columns = [c for c in raw_candidate_columns if c not in CATEGORICAL_COLUMNS]
print(f"Avenue 2's non-management environmental candidate pool: {len(raw_candidate_columns)} raw columns")
print(f"  -> {len(candidate_columns)} after excluding categorical class IDs (screened separately, not here)")
print(candidate_columns)

table, model_columns = prepare_broad_table(cohort, raw_candidate_columns)
print(f"\n{len(table):,} complete-case plots for the {cohort} cohort")
print(f"Target column: {TARGET}")

Avenue 2's non-management environmental candidate pool: 33 raw columns
  -> 30 after excluding categorical class IDs (screened separately, not here)
['elevation', 'slope_degrees', 'northness', 'eastness', 'profile_curvature', 'plan_curvature', 'tpi', 'elevation_roughness', 'ceh_twi', 'solar_radiation_index', 'frost_hollow_flag', 'topex', 'windward_topex', 'whcl', 'gwa_wind_speed_50m', 'tpi_500m', 'local_relief_500m', 'tas_mean', 'groundfrost_mean', 'chelsa_bio1_celsius', 'chelsa_gdd5_degc', 'chelsa_bio12_precip_mm', 'soilgrids_ph', 'dist_to_watercourse', 'dist_to_cpmt_boundary', 'dist_to_forest_perimeter', 'dist_to_scpt_boundary', 'dist_to_block_boundary', 'cpmt_compactness_ratio', 'dist_to_road']
  Rows before filtering: 287,064
  Removed by Age < 30 at the 2023 survey (whole plot dropped): 54,616 rows
  Removed by yield class between 2 and 50 (whole plot dropped): 0 rows
  Rows after filtering: 232,448


  Rows before filtering: 287,064
  Removed by Age < 30 at the 2023 survey (whole plot dropped): 54,616 rows
  Removed by yield class between 2 and 50 (whole plot dropped): 0 rows
  Rows after filtering: 232,448
  Disturbance cleaning: excluded 315 of 56,841 plots (clearfell-like or measurement-inconsistent) before fitting y_max


  Broad complete-case population: dropped 412 of 56,526 plots

56,114 complete-case plots for the 4survey cohort
Target column: local_y_max_difference


## 2. Run the shared screen on Avenue 2's own candidate pool

Same three-stage procedure as the Avenue 1 notebook (deterministic duplicates, near-exact
empirical duplicates, iterative VIF) -- full screen, not the dedup-only version, since Avenue 2's
actual models (Elastic Net, XGBoost) are exactly the case the `llm-council` review said this full
treatment belongs to: collinearity there directly corrupts coefficients/SHAP attribution, not
just adding harmless redundant compute the way a raw PINN input would.

In [3]:
kept, drop_log, flags = run_full_screen(table, candidate_columns, FEATURE_PROVENANCE, target_column=TARGET)
print(f"{len(candidate_columns)} candidates -> {len(kept)} kept\n")
print(drop_log.to_string(index=False) if len(drop_log) else "(nothing dropped)")

30 candidates -> 20 kept

                column                  stage                                                              reason
   chelsa_bio1_celsius 2_near_exact_duplicate                                     rho=0.997 with its kept partner
             elevation                  3_vif VIF=38.29 against the other 28 remaining candidates (threshold 5.0)
 dist_to_cpmt_boundary                  3_vif VIF=20.64 against the other 27 remaining candidates (threshold 5.0)
   elevation_roughness                  3_vif VIF=13.12 against the other 26 remaining candidates (threshold 5.0)
      chelsa_gdd5_degc                  3_vif VIF=11.52 against the other 25 remaining candidates (threshold 5.0)
dist_to_block_boundary                  3_vif  VIF=7.47 against the other 24 remaining candidates (threshold 5.0)
                   tpi                  3_vif  VIF=6.68 against the other 23 remaining candidates (threshold 5.0)
                 topex                  3_vif  VIF=6.04 agains

In [4]:
print("Spatial-confound flags (informational only, nothing auto-dropped):\n")
print(flags.to_string(index=False))

Spatial-confound flags (informational only, nothing auto-dropped):

                  column  raw_vif  compartment_residualized_vif  likely_spatial_confound_not_true_duplicate
      gwa_wind_speed_50m 4.665593                      2.355910                                       False
                tas_mean 4.290085                      1.572119                                       False
       local_relief_500m 3.860301                      1.138520                                       False
          windward_topex 3.418772                      1.856656                                       False
                tpi_500m 3.174167                      3.358275                                       False
            dist_to_road 2.714010                      1.439198                                       False
           slope_degrees 2.643771                      1.457376                                       False
                 ceh_twi 2.558949                      2.386695     

## 3. Compare against what Avenue 2 already does

Avenue 2 has its own, already-existing, empirically-driven representation-selection
infrastructure -- `run_wind_height_swap_check.py` and `run_representation_cv_check.py` -- which
tested candidate wind/terrain representations directly under spatial cross-validation (real
held-out predictive performance), not just correlation. `FINAL_FEATURE_COLUMNS`
(`SCOPE_GROUPS["terrain_wind"]`) is the result: it already swaps 10m for 50m wind and already
excludes `inverse_slope_proxy` (an exact duplicate) and `tpi_250m` (never included at all,
citing the same redundancy this project's shared screen also finds). That is a **more direct**
validation method than a correlation/VIF screen for the specific representation choices it
covers -- this section checks whether the two methods agree, not which one to trust more.

In [5]:
av2_terrain_wind_set = set(FINAL_FEATURE_COLUMNS)
screen_terrain_wind_candidates = [c for c in candidate_columns if c in TERRAIN_AND_WIND_COLUMNS or c in SWAPPED_COLUMNS]
screen_kept_terrain_wind = [c for c in kept if c in screen_terrain_wind_candidates]

print("Avenue 2's own terrain_wind (FINAL_FEATURE_COLUMNS), validated via spatial-CV representation checks:")
print(sorted(av2_terrain_wind_set))
print(f"\nShared screen's terrain/wind survivors from the SAME wider candidate pool (VIF/correlation only):")
print(sorted(screen_kept_terrain_wind))

only_in_av2 = av2_terrain_wind_set - set(screen_kept_terrain_wind)
only_in_screen = set(screen_kept_terrain_wind) - av2_terrain_wind_set
print(f"\nIn Avenue 2's set but not flagged as a screen survivor: {sorted(only_in_av2)}")
print(f"In the screen's survivors but not in Avenue 2's set: {sorted(only_in_screen)}")

Avenue 2's own terrain_wind (FINAL_FEATURE_COLUMNS), validated via spatial-CV representation checks:
['ceh_twi', 'eastness', 'elevation', 'elevation_roughness', 'frost_hollow_flag', 'gwa_wind_speed_50m', 'local_relief_500m', 'northness', 'plan_curvature', 'profile_curvature', 'slope_degrees', 'solar_radiation_index', 'topex', 'tpi', 'tpi_500m', 'whcl', 'windward_topex']

Shared screen's terrain/wind survivors from the SAME wider candidate pool (VIF/correlation only):
['ceh_twi', 'eastness', 'frost_hollow_flag', 'gwa_wind_speed_50m', 'northness', 'plan_curvature', 'profile_curvature', 'slope_degrees', 'whcl', 'windward_topex']

In Avenue 2's set but not flagged as a screen survivor: ['elevation', 'elevation_roughness', 'local_relief_500m', 'solar_radiation_index', 'topex', 'tpi', 'tpi_500m']
In the screen's survivors but not in Avenue 2's set: []


## Findings

- **Target-circularity check passes** for Avenue 2's own target, independently re-confirmed.
- Avenue 2's `terrain_wind` set was already built through a genuinely rigorous process --
  `run_wind_height_swap_check.py`/`run_representation_cv_check.py` tested real out-of-fold
  spatial-CV performance for competing representations (10m vs. 50m wind; with/without
  `tpi_250m`; with/without `local_relief_500m`), not just correlation. That's a stronger form of
  evidence than what this correlation/VIF screen alone provides for the specific choices it
  covers.
- The comparison above shows whether the two independent methods (Avenue 2's spatial-CV
  representation checks vs. this session's VIF/correlation screen) agree on which columns
  survive -- see the printed set differences. Where they agree, that's convergent evidence from
  two different methods. Where they disagree, it's worth understanding why before treating either
  one as final (e.g. a variable a spatial-CV check found genuinely predictive despite moderate
  VIF might be capturing something the linear collinearity measure doesn't).
- **`management` was excluded from this screen on purpose**, not because it's untrustworthy --
  Avenue 2's own `SCOPE_GROUPS["terrain_wind_plus_management"]`/`["broad_environment_plus_management"]`
  already test its marginal contribution directly, which is the right way to answer that
  particular question (an architectural comparison), not a multicollinearity screen.
- **No Avenue 2 file was modified and no Avenue 2 model was retrained** to produce any number in
  this notebook.

See the companion notebook `multicollinearity_screen_av1.ipynb` for the equivalent Avenue 1
analysis, and `documentation/experiment_log.md`'s 2026-08-06 entry for the `llm-council` review
this shared method responds to.